In [ ]:
!pip uninstall -y torchvision
!pip install torchvision==0.20.1+cu130 --index-url https://download.pytorch.org/whl/cu130
!pip install -q evaluate

Looking in indexes: https://download.pytorch.org/whl/cu130
ERROR: Could not find a version that satisfies the requirement torchvision==0.20.1+cu130 (from versions: 0.1.6, 0.2.0, 0.24.0+cu130, 0.24.1+cu130, 0.25.0+cu130, 0.26.0+cu130)
ERROR: No matching distribution found for torchvision==0.20.1+cu130


In [9]:
# =========================================
# 1️⃣ Install Libraries
# =========================================
!pip install -U transformers datasets seqeval torch tqdm

# =========================================
# 2️⃣ Imports
# =========================================
import json
import re
import torch
import numpy as np
from tqdm import tqdm
import evaluate
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, pipeline
from sklearn.model_selection import train_test_split

seqeval_metric = evaluate.load("seqeval")

# =========================================
# 3️⃣ Load Dataset
# =========================================
dataset_path = "/content/OS-cleaned_file.jsonl"

dataset = []
with open(dataset_path, "r") as f:
    for line in f:
        dataset.append(json.loads(line))

texts = [d["input"] for d in dataset]
labels = [json.loads(d["output"]) for d in dataset]

# =========================================
# 4️⃣ BIO Label Setup
# =========================================
entity_types = ["TOPIC", "CONCEPT", "METHOD", "ALGORITHM"]

label_list = ["O"]
for t in entity_types:
    label_list.append(f"B-{t}")
    label_list.append(f"I-{t}")

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {v: k for k, v in label2id.items()}

# =========================================
# 5️⃣ Text Normalization
# =========================================
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s\-]", "", text)
    return text

# =========================================
# 6️⃣ Encode Labels (BIO FIXED)
# =========================================
def encode_labels(text, entities):
    tokens = text.split()
    token_labels = ["O"] * len(tokens)

    for ent in entities:
        ent_text = normalize(ent["entity"])
        ent_tokens = ent_text.split()
        ent_label = ent["label"] if ent["label"] in entity_types else "CONCEPT"

        for i in range(len(tokens) - len(ent_tokens) + 1):
            if tokens[i:i+len(ent_tokens)] == ent_tokens:
                token_labels[i] = f"B-{ent_label}"
                for j in range(1, len(ent_tokens)):
                    token_labels[i+j] = f"I-{ent_label}"

    return token_labels

# Normalize texts
texts = [normalize(t) for t in texts]

token_labels = [encode_labels(t, e) for t, e in zip(texts, labels)]

# =========================================
# 7️⃣ Split Dataset
# =========================================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, token_labels, test_size=0.15, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.33, random_state=42
)

train_dataset = Dataset.from_dict({"tokens": [t.split() for t in train_texts], "ner_tags": train_labels})
val_dataset = Dataset.from_dict({"tokens": [t.split() for t in val_texts], "ner_tags": val_labels})
test_dataset = Dataset.from_dict({"tokens": [t.split() for t in test_texts], "ner_tags": test_labels})

# =========================================
# 8️⃣ Models
# =========================================
scibert_name = "allenai/scibert_scivocab_uncased"
roberta_name = "roberta-base"

scibert_tokenizer = AutoTokenizer.from_pretrained(scibert_name)
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)

scibert_model = AutoModelForTokenClassification.from_pretrained(scibert_name, num_labels=len(label_list))
roberta_model = AutoModelForTokenClassification.from_pretrained(roberta_name, num_labels=len(label_list))

for model in [scibert_model, roberta_model]:
    model.config.id2label = id2label
    model.config.label2id = label2id

# =========================================
# 9️⃣ Tokenization
# =========================================
def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            else:
                label_ids.append(label2id[label[word_id]])

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

train_sci = train_dataset.map(lambda x: tokenize_and_align_labels(x, scibert_tokenizer), batched=True)
val_sci = val_dataset.map(lambda x: tokenize_and_align_labels(x, scibert_tokenizer), batched=True)

train_rob = train_dataset.map(lambda x: tokenize_and_align_labels(x, roberta_tokenizer), batched=True)
val_rob = val_dataset.map(lambda x: tokenize_and_align_labels(x, roberta_tokenizer), batched=True)

for d in [train_sci, val_sci, train_rob, val_rob]:
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# =========================================
# 🔟 Training
# =========================================
args = TrainingArguments(
    output_dir="./models",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    eval_strategy="epoch",   # ✅ FIXED
    save_strategy="epoch",
    logging_dir="./logs"
)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    labels = p.label_ids

    true = [[id2label[l] for l in lab if l != -100] for lab in labels]
    pred = [[id2label[p] for p, l in zip(pr, lab) if l != -100] for pr, lab in zip(preds, labels)]

    return seqeval_metric.compute(predictions=pred, references=true)

trainer_sci = Trainer(model=scibert_model, args=args, train_dataset=train_sci, eval_dataset=val_sci, compute_metrics=compute_metrics)
trainer_rob = Trainer(model=roberta_model, args=args, train_dataset=train_rob, eval_dataset=val_rob, compute_metrics=compute_metrics)

trainer_sci.train()
trainer_rob.train()

# =========================================
# 1️⃣1️⃣ Pipelines
# =========================================
sci_pipe = pipeline("ner", model=scibert_model, tokenizer=scibert_tokenizer, aggregation_strategy="simple")
rob_pipe = pipeline("ner", model=roberta_model, tokenizer=roberta_tokenizer, aggregation_strategy="simple")

# =========================================
# 1️⃣2️⃣ Strong Filtering
# =========================================
STOPWORDS = {"the", "and", "or", "to", "of", "in", "a"}

def clean_entity(text):
    text = text.lower().strip()
    if text in STOPWORDS:
        return None
    if len(text) < 3:
        return None
    if text.endswith(("and", "or")):
        return None
    if "," in text:
        return None
    return text

# =========================================
# 1️⃣3️⃣ Ensemble (IMPROVED)
# =========================================
def ensemble(sci, rob):
    combined = {}

    for e in sci:
        key = clean_entity(e["word"])
        if not key: continue
        combined[key] = {"label": e["entity_group"], "score": e["score"], "source": "sci"}

    for e in rob:
        key = clean_entity(e["word"])
        if not key: continue

        if key in combined:
            combined[key]["score"] = max(combined[key]["score"], e["score"])
            combined[key]["source"] = "both"
        else:
            if e["score"] > 0.75:
                combined[key] = {"label": e["entity_group"], "score": e["score"], "source": "rob"}

    # keep strong ones
    final = []
    for k, v in combined.items():
        if v["source"] == "both" or v["score"] > 0.8:
            final.append({
                "entity": k,
                "label": v["label"],
                "score": round(float(v["score"]), 3)
            })

    return final

# =========================================
# 1️⃣4️⃣ Run Ensemble
# =========================================
results = []

for text in tqdm(test_texts):
    sci = sci_pipe(text)
    rob = rob_pipe(text)

    ents = ensemble(sci, rob)

    results.append({
        "text": text,
        "entities": ents
    })

# =========================================
# 1️⃣5️⃣ Save
# =========================================
with open("FINAL_KG_READY.json", "w") as f:
    json.dump(results, f, indent=4)

print("✅ CLEAN KG FILE READY")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Map:   0%|          | 0/1327 [00:00<?, ? examples/s]

Map:   0%|          | 0/157 [00:00<?, ? examples/s]

Map:   0%|          | 0/1327 [00:00<?, ? examples/s]

...

{'eval_loss': '0.06977', 'eval_ALGORITHM': {'precision': 0.6190476190476191, 'recall': 0.9285714285714286, 'f1': 0.742857142857143, 'number': 28}, 'eval_CONCEPT': {'precision': 0.8899082568807339, 'recall': 0.9336898395721925, 'f1': 0.9112734864300627, 'number': 935}, 'eval_METHOD': {'precision': 0.6730769230769231, 'recall': 0.6481481481481481, 'f1': 0.6603773584905661, 'number': 54}, 'eval_TOPIC': {'precision': 0.8961937716262975, 'recall': 0.9810606060606061, 'f1': 0.9367088607594936, 'number': 264}, 'eval_overall_precision': '0.8746', 'eval_overall_recall': '0.9313', 'eval_overall_f1': '0.9021', 'eval_overall_accuracy': '0.982', 'eval_runtime': '1.513', 'eval_samples_per_second': '103.8', 'eval_steps_per_second': '6.611', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '243.8', 'train_samples_per_second': '27.21', 'train_steps_per_second': '1.702', 'train_loss': '0.1137', 'epoch': '5'}


100%|██████████| 78/78 [00:02<00:00, 28.70it/s]

✅ CLEAN KG FILE READY


In [10]:
import json
import re

# =========================================
# 1️⃣ Load file
# =========================================
with open("FINAL_KG_READY.json", "r") as f:
    data = json.load(f)

# =========================================
# 2️⃣ Stopwords & rules
# =========================================
STOPWORDS = {
    "the", "and", "or", "to", "of", "in", "a", "an",
    "is", "are", "was", "were", "be", "been", "being"
}

MIN_LEN = 3
MIN_SCORE = 0.75

# =========================================
# 3️⃣ Clean text
# =========================================
def clean_text(text):
    text = text.lower().strip()

    # remove subword markers
    text = text.replace("##", "")

    # remove weird chars
    text = re.sub(r"[^\w\s\-]", "", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# =========================================
# 4️⃣ Validate entity
# =========================================
def is_valid(entity, score):
    words = entity.split()

    if score < MIN_SCORE:
        return False

    if len(entity) < MIN_LEN:
        return False

    if entity in STOPWORDS:
        return False

    if all(w in STOPWORDS for w in words):
        return False

    # remove broken endings
    if entity.endswith(("and", "or")):
        return False

    # remove numeric junk
    if entity.isdigit():
        return False

    return True

# =========================================
# 5️⃣ Merge subword fragments (IMPORTANT)
# =========================================
def merge_fragments(entities):
    merged = []
    buffer = ""

    for ent in entities:
        word = ent["entity"]

        # if it's a fragment (very short or weird)
        if len(word) <= 3:
            buffer += word
        else:
            if buffer:
                merged.append({
                    "entity": buffer + word,
                    "label": ent["label"],
                    "score": ent["score"]
                })
                buffer = ""
            else:
                merged.append(ent)

    return merged

# =========================================
# 6️⃣ Main cleaning
# =========================================
cleaned_data = []

for item in data:
    seen = set()
    cleaned_entities = []

    # merge fragments first
    merged_entities = merge_fragments(item["entities"])

    for ent in merged_entities:
        text = clean_text(ent["entity"])
        score = float(ent.get("score", 1.0))
        label = ent["label"]

        if not is_valid(text, score):
            continue

        # deduplicate
        key = text
        if key in seen:
            continue
        seen.add(key)

        cleaned_entities.append({
            "entity": text,
            "label": label,
            "score": round(score, 3)
        })

    cleaned_data.append({
        "text": item["text"],
        "entities": cleaned_entities
    })

# =========================================
# 7️⃣ Save final file
# =========================================
with open("FINAL_CLEANED_KG.json", "w") as f:
    json.dump(cleaned_data, f, indent=4)

print("🔥 FINAL CLEAN KG FILE READY → FINAL_CLEANED_KG.json")

🔥 FINAL CLEAN KG FILE READY → FINAL_CLEANED_KG.json


In [6]:
!rm -rf /content/ner_models

In [ ]:
# =========================================
# 1️⃣ Install Libraries
# =========================================
!pip install -q transformers datasets scikit-learn

# =========================================
# 2️⃣ Imports
# =========================================
import json
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    logging
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

logging.set_verbosity_error()

# =========================================
# 3️⃣ Load RE Dataset
# =========================================
with open("/content/CN-dataset_relations_fixed.json") as f:
    data = json.load(f)

texts = [d["input"] for d in data]
labels = [d["output"].strip() for d in data]

label_list = ["used_for", "based_on", "implements", "part_of", "improves", "no_relation"]
label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

labels = [l if l in label2id else "no_relation" for l in labels]

# =========================================
# 4️⃣ Train / Val / Test Split
# =========================================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42, stratify=labels
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

def make_dataset(texts, labels):
    return Dataset.from_dict({
        "text": texts,
        "label": [label2id[l] for l in labels]
    })

train_dataset = make_dataset(train_texts, train_labels)
val_dataset = make_dataset(val_texts, val_labels)
test_dataset = make_dataset(test_texts, test_labels)

# =========================================
# 5️⃣ Tokenization
# =========================================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

for d in [train_dataset, val_dataset, test_dataset]:
    d.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =========================================
# 6️⃣ Model
# =========================================
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# =========================================
# 7️⃣ Metrics
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# =========================================
# 8️⃣ Training Args
# =========================================
training_args = TrainingArguments(
    output_dir="./roberta-relation",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# =========================================
# 9️⃣ Trainer
# =========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# =========================================
# 🔟 Train
# =========================================
trainer.train()

# =========================================
# 🧪 TEST SET EVALUATION
# =========================================
test_metrics = trainer.evaluate(eval_dataset=test_dataset)

print("\n📊 TEST SET RESULTS:")
print(f"Accuracy : {test_metrics['eval_accuracy']:.4f}")
print(f"Precision: {test_metrics['eval_precision']:.4f}")
print(f"Recall   : {test_metrics['eval_recall']:.4f}")
print(f"F1 Score : {test_metrics['eval_f1']:.4f}")

# =========================================
# 1️⃣1️⃣ Load NER Output
# =========================================
with open("/content/ensemble_entities.json") as f:
    test_data = json.load(f)

# =========================================
# 1️⃣2️⃣ Clean Entities
# =========================================
def clean_entities(entities):
    cleaned = []
    for e in entities:
        text = e["entity"].strip().lower()

        if len(text) < 4:
            continue
        if text in ["the", "this", "that", "data"]:
            continue
        if e["label"] == "OTHER":
            continue

        cleaned.append(e)

    return cleaned[:5]   # 🔥 LIMIT TO TOP 5

# =========================================
# 1️⃣3️⃣ Direction-Aware Prediction
# =========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def predict_best_relation(e1, t1, e2, t2, text):
    def run(inp):
        inputs = tokenizer(inp, return_tensors="pt", truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model(**inputs)
            probs = torch.softmax(out.logits, dim=1)
            conf, pred = torch.max(probs, dim=1)
        return id2label[pred.item()], conf.item()

    input1 = f"Sentence: {text} Entity1: {e1} ({t1}). Entity2: {e2} ({t2})."
    input2 = f"Sentence: {text} Entity1: {e2} ({t2}). Entity2: {e1} ({t1})."

    r1, c1 = run(input1)
    r2, c2 = run(input2)

    if c1 >= c2:
        return e1, t1, r1, e2, t2, c1
    else:
        return e2, t2, r2, e1, t1, c2

# =========================================
# 1️⃣4️⃣ Predict Relations
# =========================================
triplets = []
seen = set()

for item in test_data:
    text = item["text"]
    entities = clean_entities(item.get("entities", []))

    for i in range(len(entities)):
        for j in range(i+1, len(entities)):

            e1 = entities[i]["entity"]
            t1 = entities[i]["label"]
            e2 = entities[j]["entity"]
            t2 = entities[j]["label"]

            if e1.lower() == e2.lower():
                continue

            e1, t1, rel, e2, t2, conf = predict_best_relation(e1, t1, e2, t2, text)

            # 🔥 Strong filtering
            if rel == "no_relation" or conf < 0.8:
                continue

            key = (e1.lower(), rel, e2.lower())
            if key in seen:
                continue
            seen.add(key)

            triplets.append({
                "text": text,
                "entity1": e1,
                "entity1_type": t1,
                "relation": rel,
                "entity2": e2,
                "entity2_type": t2,
                "confidence": round(conf, 3)
            })

# =========================================
# 1️⃣5️⃣ Save Output
# =========================================
with open("final_triplets_level2.json", "w") as f:
    json.dump(triplets, f, indent=4)

print("\n✅ FINAL LEVEL-2 triplets saved to final_triplets_level2.json")

Map:   0%|          | 0/1756 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'eval_loss': '0.9263', 'eval_accuracy': '0.6837', 'eval_precision': '0.6133', 'eval_recall': '0.6837', 'eval_f1': '0.6456', 'eval_runtime': '1.312', 'eval_samples_per_second': '74.67', 'eval_steps_per_second': '9.905', 'epoch': '1'}


...

{'eval_loss': '0.7162', 'eval_accuracy': '0.8571', 'eval_precision': '0.8595', 'eval_recall': '0.8571', 'eval_f1': '0.8513', 'eval_runtime': '1.297', 'eval_samples_per_second': '75.54', 'eval_steps_per_second': '10.02', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.4658', 'grad_norm': '34.85', 'learning_rate': '1.836e-06', 'epoch': '4.545'}
{'eval_loss': '0.6796', 'eval_accuracy': '0.8571', 'eval_precision': '0.8588', 'eval_recall': '0.8571', 'eval_f1': '0.854', 'eval_runtime': '1.295', 'eval_samples_per_second': '75.66', 'eval_steps_per_second': '10.04', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '469.6', 'train_samples_per_second': '18.7', 'train_steps_per_second': '2.342', 'train_loss': '0.647', 'epoch': '5'}
{'eval_loss': '0.7427', 'eval_accuracy': '0.7959', 'eval_precision': '0.7974', 'eval_recall': '0.7959', 'eval_f1': '0.7952', 'eval_runtime': '1.3', 'eval_samples_per_second': '75.4', 'eval_steps_per_second': '10', 'epoch': '5'}

📊 TEST SET RESULTS:
Accuracy : 0.7959
Precision: 0.7974
Recall   : 0.7959
F1 Score : 0.7952

✅ FINAL LEVEL-2 triplets saved to final_triplets_level2.json
